In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ==========================================
# 1. THE CUSTOMER SERVICE STATE
# ==========================================
class SupportState(TypedDict):
    customer_msg: str
    issue_type: str   # "billing", "delivery", or "unknown"
    agent_reply: str

# Nodes mimicking steps in the ticketing system
def read_ticket_node(state: SupportState):
    return {"customer_msg": "I was charged twice for my premium upgrade!"}

def automated_classifier_node(state: SupportState):
    print("🤖 [Classifier] Scanning keywords...")
    # Oops! The simulated AI made a mistake and misclassified the issue!
    return {"issue_type": "delivery"}

def support_router(state: SupportState):
    if state["issue_type"] == "billing":
        return "billing_department"
    else:
        return "shipping_department"

def billing_node(state: SupportState):
    return {"agent_reply": "We are so sorry for the double charge. We have initiated an instant refund."}

def shipping_node(state: SupportState):
    return {"agent_reply": "Your package is on its way. Please allow up to 48 hours for delivery."}

# Build the layout map
builder = StateGraph(SupportState)
builder.add_node("read_ticket", read_ticket_node)
builder.add_node("classify", automated_classifier_node)
builder.add_node("billing_dept", billing_node)
builder.add_node("shipping_dept", shipping_node)

builder.add_edge(START, "read_ticket")
builder.add_edge("read_ticket", "classify")

# Add conditional path after classification
builder.add_conditional_edges(
    "classify",
    support_router,
    {
        "billing_department": "billing_dept",
        "shipping_department": "shipping_dept"
    }
)
builder.add_edge("billing_dept", END)
builder.add_edge("shipping_dept", END)

# Compile with in-memory persistence track
checkpointer = MemorySaver()
app = builder.compile(checkpointer=checkpointer)

# ==========================================
# 2. RUN 1: Watch the System Fail
# ==========================================
print("=== FIRST RUN: Automated Pipeline Execution ===")
config = {"configurable": {"thread_id": "ticket_abhishek_001"}}

result_1 = app.invoke({}, config)
print(f"❌ Initial Classification Result: {result_1['issue_type']}")
print(f"❌ Wrong Automated Response: '{result_1['agent_reply']}'")

# ==========================================
# 3. THE TIME MACHINE: Inspecting History
# ==========================================
print("\n=== STEPPING INTO THE TIME MACHINE ===")

# Pull the complete list of chronological snapshots from the database for this thread
history = list(app.get_state_history(config))

print(f"Found {len(history)} individual checkpoint saves in the database.")
for index, state_snapshot in enumerate(history):
    print(f"   Checkpoint Index [{index}] -> Next Node slated to run: {state_snapshot.next} | State Values: {state_snapshot.values}")

# We want to target the state *right after* 'read_ticket' finished, but *before* the router executed!
# Looking at our prints, that is snapshot index 2 or 3 depending on execution depth. 
# Let's locate the checkpoint right after 'classify' completed, but before the wrong routing finished.
target_state = history[1] # The state snapshot captured right at the mistake gate
print(f"\n🎯 Selected Target Checkpoint ID to Fork: {target_state.config['configurable']['checkpoint_id']}")



=== FIRST RUN: Automated Pipeline Execution ===
🤖 [Classifier] Scanning keywords...
❌ Initial Classification Result: delivery
❌ Wrong Automated Response: 'Your package is on its way. Please allow up to 48 hours for delivery.'

=== STEPPING INTO THE TIME MACHINE ===
Found 5 individual checkpoint saves in the database.
   Checkpoint Index [0] -> Next Node slated to run: () | State Values: {'customer_msg': 'I was charged twice for my premium upgrade!', 'issue_type': 'delivery', 'agent_reply': 'Your package is on its way. Please allow up to 48 hours for delivery.'}
   Checkpoint Index [1] -> Next Node slated to run: ('shipping_dept',) | State Values: {'customer_msg': 'I was charged twice for my premium upgrade!', 'issue_type': 'delivery'}
   Checkpoint Index [2] -> Next Node slated to run: ('classify',) | State Values: {'customer_msg': 'I was charged twice for my premium upgrade!'}
   Checkpoint Index [3] -> Next Node slated to run: ('read_ticket',) | State Values: {}
   Checkpoint Index [

In [2]:
# ==========================================
# 4. FORKING THE HISTORY: Override the past
# ==========================================
print("\n✍️ Rewriting history: Modifying database value to 'billing'...")

# We use update_state, but we pass the target_state.config (which carries the specific old checkpoint ID!)
forked_config = app.update_state(
    target_state.config,
    values={"issue_type": "billing"}, # Force overwrite the mistake
    as_node="classify"
)

# ==========================================
# 5. RESUMING FROM THE FORK
# ==========================================
print("🔄 Re-executing thread from the modified past...")
# We pass None and the new forked_config matrix returned by update_state
final_forked_result = app.invoke(None, config=forked_config)

print(f"\n🏁 New Branched Result: {final_forked_result['issue_type']}")
print(f"🏁 Corrected Automated Response: '{final_forked_result['agent_reply']}'")


✍️ Rewriting history: Modifying database value to 'billing'...
🔄 Re-executing thread from the modified past...

🏁 New Branched Result: billing
🏁 Corrected Automated Response: 'We are so sorry for the double charge. We have initiated an instant refund.'
